# CareReturn AI — Temporal Patient Holdout

Hold out later-entering patients and purge post-cutoff training encounters.

800 synthetic patients, six encounters each. No real patient data or clinical validation.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from healthcare_ai.registry_loading import load_registry
from healthcare_ai.registry_labels import labeled_registry
from healthcare_ai.registry_split import registry_partitions
from healthcare_ai.settings import FEATURES,TARGET
from healthcare_ai.pipeline import build
from healthcare_ai.metrics import evaluate
raw=load_registry();frame=labeled_registry(raw)
train,val,test=registry_partitions(frame);model=build().fit(train[FEATURES],train[TARGET])


In [2]:
from healthcare_ai.registry_temporal import temporal_patient_split
a,b=temporal_patient_split(frame)
m=build().fit(a[FEATURES],a[TARGET])
print('Train/test patients:',a.patient_id.nunique(),b.patient_id.nunique())
print(evaluate(b[TARGET],m.predict_proba(b[FEATURES])[:,1]))

Train/test patients: 624 176
{'auroc': 0.696226534753932, 'average_precision': 0.30068443049103016, 'brier': 0.1320071696978133, 'balanced_accuracy': 0.4994292237442922, 'sensitivity': 0.0, 'specificity': 0.9988584474885844, 'tp': 0, 'tn': 875, 'fp': 1, 'fn': 180, 'n': 1056, 'threshold': 0.5}


## Interpretation

All outcomes are simulator-generated. Repeated encounters from each patient remain in one partition. Clinical, prospective and external validation have not been performed. Thresholds and probability bands are educational examples.